# 🔬 Module 02: QCL Spectral Dimensionality Reduction

**Objective:** Load raw `Daylight Solutions Spero®` hyperspectral `.mat` cubes, preprocess the Mid-IR fingerprint region (912 to 1800 cm⁻¹), and apply Manifold Learning (UMAP/PCA) to visualize phenotypic clusters without chemical staining.

In [ ]:
import os
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
from pathlib import Path

# Configuration
DATA_DIR = Path('./data/raw')
PROCESSED_DIR = Path('./data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f'Looking for .mat hyperspectral cubes in: {DATA_DIR.absolute()}')

## 1. Loader & Preprocessing Functions
Hyperspectral data cubes are essentially 3D tensors: `(x_pixels, y_pixels, spectral_bands_cm-1)`. We must unfold these into a 2D matrix `(pixels, bands)` for machine learning, while retaining the spatial maps to rebuild the image later.

In [ ]:
def load_spero_cube(mat_path):
    '''
    Load and parse a .mat file from the Spero QCL microscope.
    Expected keys:
    - 'cube': 3D Hyperspectral data (x, y, wavenumbers)
    - 'wavenumbers': 1D array of spectral bands in cm^-1
    - 'labels': (Optional) 2D array of pathologist annotations
    '''
    print(f'[INFO] Loading {mat_path.name}...')
    mat = sio.loadmat(str(mat_path))
    
    # Fallback to general introspection if specific keys vary
    cube_key = [k for k in mat.keys() if not k.startswith('_') and type(mat[k]) == np.ndarray and mat[k].ndim == 3]
    if not cube_key:
        raise ValueError('No 3D data cube found in .mat file.')
        
    cube = mat[cube_key[0]]
    print(f'[OK] Loaded cube with shape: {cube.shape}')
    return cube, mat


## 2. Dimensionality Reduction (PCA & UMAP)
Since the QCL captures hundreds of wavenumbers per pixel, the data is highly collinear. We use PCA for initial noise reduction, followed by UMAP to identify non-linear cellular phenotypes (e.g., epithelium vs. stroma).

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

def apply_pca_reduction(cube, n_components=10):
    '''
    Unfold the cube, standardize spectra, and apply PCA.
    '''
    h, w, bands = cube.shape
    X = cube.reshape((h * w, bands))
    
    # Standardize
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    pca = PCA(n_components=n_components)
    X_pca = pca.fit_transform(X_scaled)
    
    print(f'[INFO] PCA explained variance ratio ({n_components} components): {np.sum(pca.explained_variance_ratio_):.2f}')
    
    # Refold for visualization
    pca_img = X_pca.reshape((h, w, n_components))
    return pca_img, pca, scaler
